In [1]:
import os
import pandas as pd
from asset_modeling.credit import loan_portfolio, private_credit_loan_model
from data.sofr import get_sofr_data
from data.ratings import get_effective_yield

fred_api_key = os.getenv("FRED_API_KEY")

#### credit functions

##### private_credit_loan_model()
Models a single private credit loan and generates a quarterly payment schedule with amortization, interest, and fees.

**Arguments:**
- `investment_name`: Name/identifier for the investment
- `investment_date`: Date the investment/loan is initiated
- `maturity_date`: Date the loan matures
- `loan_size`: Principal amount of the loan
- `spread`: Spread in basis points
- `base_rate`: Base interest rate
- `sofr_assumption`: SOFR rate assumption
- `cash_interest_rate`: Annual cash interest rate (paid quarterly)
- `pik_interest`: Annual payment-in-kind interest rate (added to balance quarterly)
- `amortization`: Annual amortization rate applied to original loan amount
- `oid`: Original issue discount as a percentage of loan_size
- `exit_fee`: Fee as percentage of remaining balance at maturity/prepayment
- `prepayment_date`: Optional date when loan is prepaid (if None, uses maturity_date)

**Returns:** DataFrame with quarterly schedule including cash flows, balances, interest, and IRR

---

##### loan_portfolio()
Aggregates multiple loans into a portfolio summary by combining individual loan schedules and performing quarterly rollup.

**Arguments:**
- `schedule_of_investments`: DataFrame or dict with one row per loan containing all required columns for `private_credit_loan_model()`

**Returns:** DataFrame with quarterly portfolio totals (invested_amount, total_payment, remaining_balance_payment, ending_balance, beginning_balance)

---
#### Example: private_credit_loan_model() 

1) Determine Cash Rate
    - Total_Rate = SOFR + Spread = Cash Interest Rate + PIK Interest Rate
        - Example: 10 = 4 + 6 = 8 + 2
    Cash Interest Rate = SOFR + Spread - PIK Interest
        - Example: 8 = 4 + 6 - 2

2) Determine the beginning and ending balance by accounting for Amortization and PIK Interest
    Where:
        - Amortization (fixed on Par Value) = Loan Par Value * (Amortization Rate/4)
        - PIK intereest = Beginning Balance * (PIK Interest Rate/4)
        - Ending balance = Beginning Balance - Amortization + PIK Interest

3) Calculate Cash Interest
    - Ending Balance * (Cash Interest Rate/4)

4) Calculate End of Term Payments
    - Where:
        - payments are made in the last quarter when the term is over or the loan is prepaid
    - Exit Fee = Loan Par Value * Exit Fee
    - Balance Repayment = Beginning Balance at the last quarter date

5) Calculate Cash Flow
    - Where:
        - Interest and Amortization are paid quarterly
        - Balance Repayment and Exit Fees are paid in final quarter (Exit fee is based on par value)
    - Cash Payment = Interest + Amortization + Balance Repayment + Fees

In [2]:
### private_credit_loan_model()
# Example usage of the private_credit_loan_model function to create a single loan schedule
maturity_date=pd.Timestamp("2029-12-31")
rates = get_sofr_data(api_key=fred_api_key, frequency='D', end_date=maturity_date)
spreads = get_effective_yield(rating="B", api_key=fred_api_key, frequency='D', end_date=maturity_date)

loan = private_credit_loan_model(
    investment_name="Example Corp Term Loan",
    investment_date=pd.Timestamp("2024-12-31"),
    maturity_date=maturity_date,
    par_value=1_000_000,
    spread=spreads,
    base_rate='SOFR',
    sofr_rates=rates,
    # sofr_assumption=0.04,
    # sofr_floor = 0.01
    pik_interest=0.02,
    amortization=0.01,
    oid=0.02,
    exit_fee=0.02,
    prepayment_date=None,
)

print(f"Loan schedule generated with {len(loan)} quarterly periods")
loan

Loan schedule generated with 21 quarterly periods


,investment_name,quarter_end,par_value,original_investment,invested_amount,base_rate,sofr_rate,rate_status,spread,pik_rate,...,effective_yield_change,nav,contributions,distributions,ncf,cumulative_contributions,cumulative_distributions,cumulative_ncf,tvpi,irr
0,Example Corp Term Loan,2024-12-31,1000000,980000.0,980000.0,SOFR,0.0449,actual,0.0000,0.00,...,0.0000,9.800000e+05,-980000.0,0.000000e+00,-9.800000e+05,-980000.0,0.000000e+00,-980000.000000,1.000000,NaN
1,Example Corp Term Loan,2025-03-31,1000000,980000.0,0.0,SOFR,0.0441,actual,0.0733,0.02,...,0.0032,9.828448e+05,-0.0,2.685000e+04,2.685000e+04,-980000.0,2.685000e+04,-953150.000000,1.030301,0.015243
2,Example Corp Term Loan,2025-06-30,1000000,980000.0,0.0,SOFR,0.0445,actual,0.0733,0.02,...,-0.0047,9.916515e+05,-0.0,2.701113e+04,2.701113e+04,-980000.0,5.386112e+04,-926138.875000,1.066850,0.022397
3,Example Corp Term Loan,2025-09-30,1000000,980000.0,0.0,SOFR,0.0424,actual,0.0733,0.02,...,-0.0077,9.956329e+05,-0.0,2.654492e+04,2.654492e+04,-980000.0,8.040605e+04,-899593.950937,1.097999,0.102111
4,Example Corp Term Loan,2025-12-31,1000000,980000.0,0.0,SOFR,0.0387,actual,0.0733,0.02,...,-0.0062,9.951697e+05,-0.0,2.567336e+04,2.567336e+04,-980000.0,1.060794e+05,-873920.587000,1.123724,0.103241
5,Example Corp Term Loan,2026-03-31,1000000,980000.0,0.0,SOFR,0.0368,actual,0.0733,0.02,...,0.0022,9.878723e+05,-0.0,2.525195e+04,2.525195e+04,-980000.0,1.313314e+05,-848668.641987,1.142045,0.098970
6,Example Corp Term Loan,2026-06-30,1000000,980000.0,0.0,SOFR,0.0362,actual,0.0733,0.02,...,-0.0002,9.912613e+05,-0.0,2.515750e+04,2.515750e+04,-980000.0,1.564889e+05,-823511.143592,1.171174,0.102237
7,Example Corp Term Loan,2026-09-30,1000000,980000.0,0.0,SOFR,0.0362,actual,0.0733,0.02,...,-0.0002,9.922744e+05,-0.0,2.521485e+04,2.521485e+04,-980000.0,1.817037e+05,-798296.295206,1.197937,0.103469
8,Example Corp Term Loan,2026-12-31,1000000,980000.0,0.0,SOFR,0.0362,actual,0.0733,0.02,...,-0.0002,9.932874e+05,-0.0,2.527249e+04,2.527249e+04,-980000.0,2.069762e+05,-773023.810078,1.224759,0.104442
9,Example Corp Term Loan,2027-03-31,1000000,980000.0,0.0,SOFR,0.0362,actual,0.0733,0.02,...,-0.0002,9.943006e+05,-0.0,2.533041e+04,2.533041e+04,-980000.0,2.323066e+05,-747693.400024,1.251640,0.105233


In [3]:
schedule_of_investments = pd.DataFrame(
    {
        "investment_name": ["A Corp Term Loan", "B Corp Term Loan",  "C Corp Term Loan",  "D Corp Term Loan",  "E Corp Term Loan"],
        "investment_date": [pd.Timestamp("2020-03-31"), pd.Timestamp("2020-06-30"), pd.Timestamp("2020-09-30"), pd.Timestamp("2020-12-31"), pd.Timestamp("2021-03-31")],
        "maturity_date": [pd.Timestamp("2025-06-30"), pd.Timestamp("2027-06-30"), pd.Timestamp("2026-12-31"), pd.Timestamp("2028-03-31"), pd.Timestamp("2029-06-30")],
        "par_value": [1_000_000, 1_250_000, 1_000_000, 1_050_000, 1_600_000],
        # "spread": [0.07, 0.06, 0.06, 0.08, 0.07],
        "spread": ['actual', 'actual', 'actual', 'actual', 'actual'],
        "base_rate": ['SOFR', 'SOFR', 'SOFR', 'SOFR', 'SOFR'],
        "sofr_assumption": ['actual', 'actual', 'actual', 'actual', 'actual'],
        "pik_interest": [0.02, 0.02, 0.02, 0.02, 0.02],
        "amortization": [0.01, 0.01, 0.01, 0.01, 0.01],
        "oid": [0.03, 0.02, 0.03, 0.04, 0.02],
        "exit_fee": [0.02, 0.02, 0.02, 0.02, 0.02],
        "prepayment_date": [None, None, None, None, None],
    }
)

portfolio, funds, funds_summary = loan_portfolio(schedule_of_investments, rates, spreads)

In [4]:
portfolio.head()

,quarter_end,invested_amount,amortization,cash_interest,fees,remaining_balance_payment,total_payment,ending_balance,beginning_balance,nav,contributions,distributions,ncf,cumulative_contributions,cumulative_distributions,cumulative_ncf,irr,tvpi
0,2020-03-31,970000.0,0.0,0.000000,0.0,0.0,-9.700000e+05,1.000000e+06,1.000000e+06,9.700000e+05,-970000.0,0.000000,-9.700000e+05,-970000.0,0.000000,-9.700000e+05,None,1.000000
1,2020-06-30,1225000.0,2500.0,20225.000000,0.0,0.0,-1.202275e+06,2.252500e+06,2.250000e+06,2.230526e+06,-1225000.0,22725.000000,-1.202275e+06,-2195000.0,22725.000000,-2.172275e+06,0.060053,1.026538
2,2020-09-30,970000.0,5625.0,36131.687500,0.0,0.0,-9.282433e+05,3.258138e+06,3.252500e+06,3.231253e+06,-970000.0,41756.687500,-9.282433e+05,-3165000.0,64481.687500,-3.100518e+06,0.041084,1.041306
3,2020-12-31,1008000.0,8125.0,46365.689375,0.0,0.0,-9.535093e+05,4.316303e+06,4.308138e+06,4.294103e+06,-1008000.0,54490.689375,-9.535093e+05,-4173000.0,118972.376875,-4.054028e+06,0.037521,1.057531
4,2021-03-31,1568000.0,10750.0,52738.259844,0.0,0.0,-1.504512e+06,5.927135e+06,5.916303e+06,5.866592e+06,-1568000.0,63488.259844,-1.504512e+06,-5741000.0,182460.636719,-5.558539e+06,0.121389,1.053658


In [5]:
funds.head()

,investment_name,quarter_end,par_value,original_investment,invested_amount,base_rate,sofr_rate,rate_status,spread,pik_rate,...,effective_yield_change,nav,contributions,distributions,ncf,cumulative_contributions,cumulative_distributions,cumulative_ncf,tvpi,irr
0,A Corp Term Loan,2020-03-31,1000000,970000.0,970000.0,SOFR,0.0001,actual,0.0000,0.00,...,0.0000,9.700000e+05,-970000.0,0.000000,-970000.000000,-970000.0,0.000000,-970000.000000,1.000000,NaN
1,A Corp Term Loan,2020-06-30,1000000,970000.0,0.0,SOFR,0.0010,actual,0.0999,0.02,...,-0.0298,1.005526e+06,-0.0,22725.000000,22725.000000,-970000.0,22725.000000,-947275.000000,1.060053,0.029929
2,A Corp Term Loan,2020-09-30,1000000,970000.0,0.0,SOFR,0.0008,actual,0.0999,0.02,...,-0.0398,1.016789e+06,-0.0,22725.437500,22725.437500,-970000.0,45450.437500,-924549.562500,1.095092,0.031443
3,A Corp Term Loan,2020-12-31,1000000,970000.0,0.0,SOFR,0.0007,actual,0.0999,0.02,...,-0.0543,1.032487e+06,-0.0,22751.001875,22751.001875,-970000.0,68201.439375,-901798.560625,1.134731,0.139501
4,A Corp Term Loan,2021-03-31,1000000,970000.0,0.0,SOFR,0.0001,actual,0.0999,0.02,...,-0.0529,1.032634e+06,-0.0,22650.751250,22650.751250,-970000.0,90852.190625,-879147.809375,1.158233,0.130544


In [6]:
funds_summary

,investment_name,investment_date,maturity_date,par_value,spread,base_rate,sofr_assumption,pik_interest,amortization,oid,exit_fee,prepayment_date,total_payment,irr
0,A Corp Term Loan,2020-03-31,2025-06-30,1000000,actual,SOFR,actual,0.02,0.01,0.03,0.02,None,7.412739e+05,0.141755
1,B Corp Term Loan,2020-06-30,2027-06-30,1250000,actual,SOFR,actual,0.02,0.01,0.02,0.02,None,9.749669e+05,0.109353
2,C Corp Term Loan,2020-09-30,2026-12-31,1000000,actual,SOFR,actual,0.02,0.01,0.03,0.02,None,6.491814e+05,0.102894
3,D Corp Term Loan,2020-12-31,2028-03-31,1050000,actual,SOFR,actual,0.02,0.01,0.04,0.02,None,6.953255e+05,0.090575
4,E Corp Term Loan,2021-03-31,2029-06-30,1600000,actual,SOFR,actual,0.02,0.01,0.02,0.02,None,1.202886e+06,0.089356
